In [1]:
!pip install segmentation_models_pytorch -q
import os
import json
import cv2
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader, Subset
import segmentation_models_pytorch as smp
from tqdm import tqdm
import gc
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import roc_auc_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 7.6 MB/s eta 0:00:00
Running on: cuda


In [2]:
IMG_SIZE = 512
BATCH_SIZE = 16
FRACTION = 0.25 
LABEL_MAP = {"background": 0, "short sleeve top": 1, "trousers": 2, "shorts": 3, "long sleeve top": 4, "skirt": 5}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}
NUM_CLASSES = len(LABEL_MAP)

# Phase Epochs
SCRATCH_EPOCHS = 5
FINETUNE_EPOCHS = 5

In [3]:
class FashionSegDataset(Dataset):
    def __init__(self, split='train', transform=None):
        self.base_dir = f"/kaggle/input/datasets/shubhranilbasak/vr-trimmed-dataset/Trimmed-vr-dataset/DeepFashion2_Top5_{split}"
        self.img_dir = os.path.join(self.base_dir, "images")
        self.anno_dir = os.path.join(self.base_dir, "annos")
        self.filenames = [f.replace('.json', '') for f in os.listdir(self.anno_dir) if f.endswith('.json')]
        self.transform = transform

    def __len__(self): return len(self.filenames)

    def __getitem__(self, idx):
        name = self.filenames[idx]
        img = cv2.cvtColor(cv2.imread(os.path.join(self.img_dir, f"{name}.jpg")), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)
        with open(os.path.join(self.anno_dir, f"{name}.json"), 'r') as f:
            data = json.load(f)
            for key, item in data.items():
                if key.startswith('item'):
                    cat = item.get('category_name')
                    if cat in LABEL_MAP:
                        for poly in item.get('segmentation', []):
                            poly_np = np.array(poly).reshape(-1, 2).astype(np.int32)
                            cv2.fillPoly(mask, [poly_np], LABEL_MAP[cat])
        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        return img, mask.long()

# Transforms & Sampling
tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(), ToTensorV2()])
full_train = FashionSegDataset('train', transform=tf)
full_val = FashionSegDataset('val', transform=tf)

train_idx = np.random.choice(len(full_train), int(len(full_train)*FRACTION), replace=False)
val_idx = np.random.choice(len(full_val), int(len(full_val)*FRACTION), replace=False)

train_loader = DataLoader(Subset(full_train, train_idx), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(Subset(full_val, val_idx), batch_size=BATCH_SIZE)

In [4]:
# Initialize Standard U-Net (Random Weights)
model = smp.Unet(encoder_name="vgg11", encoder_weights=None, in_channels=3, classes=NUM_CLASSES).to(device)

dice_loss = smp.losses.DiceLoss(mode='multiclass')
ce_loss = nn.CrossEntropyLoss()

def criterion(pred, target):
    return dice_loss(pred, target) + ce_loss(pred, target)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3) # Higher LR for scratch

print("Starting Phase 1: Scratch Training (5 Epochs)...")
for epoch in range(SCRATCH_EPOCHS):
    model.train()
    pbar = tqdm(train_loader, desc=f"Scratch Epoch {epoch+1}")
    for imgs, masks in pbar:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=loss.item())

torch.save(model.state_dict(), "unet_scratch_checkpoint.pth")

Starting Phase 1: Scratch Training (5 Epochs)...


Scratch Epoch 5: 100%|██████████| 2253/2253 [55:40<00:00,  1.48s/it, loss=0.886]


In [5]:
model.load_state_dict(torch.load("unet_scratch_checkpoint.pth"))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5) # Lower LR for fine-tuning

print("\nStarting Phase 2: Fine-Tuning (5 Epochs)...")
for epoch in range(FINETUNE_EPOCHS):
    model.train()
    pbar = tqdm(train_loader, desc=f"Fine-Tune Epoch {epoch+1}")
    for imgs, masks in pbar:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=loss.item())

torch.save(model.state_dict(), "unet_final_best.pth")


Starting Phase 2: Fine-Tuning (5 Epochs)...


Fine-Tune Epoch 5: 100%|██████████| 2253/2253 [55:43<00:00,  1.48s/it, loss=0.961]
